Removing blank comments and duplicated comments prevents leakage prior to splitting the training, testing, and validation data.

In [ ]:
#To be copied into each notebook
#imports necessary libraries, connects to google drive COSC74 Project file, defines features, and creates helper methods
import os, re, gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report)
from statsmodels.stats.proportion import proportion_confint

from google.colab import drive
drive.mount("/content/drive")

# File locations in Drive
PROJECT_DIR = "/content/drive/MyDrive/COSC74_project"
SUBSET_PATH = f"{PROJECT_DIR}/identity_subset.parquet"     # saved by the Step 1 notebook
SPLITS_PATH = f"{PROJECT_DIR}/splits.csv"                   # saved by the Step 2 notebook
VECTORIZER_PATH = f"{PROJECT_DIR}/tfidf_vectorizer.joblib"  # saved by the Step 2 notebook

SEED = 42
IDENTITY_CATEGORIES = {
    "gender":             ["male", "female", "transgender", "other_gender"],
    "sexual_orientation": ["heterosexual", "homosexual_gay_or_lesbian", "bisexual",
                           "other_sexual_orientation"],
    "religion":           ["christian", "jewish", "muslim", "hindu", "buddhist", "atheist",
                           "other_religion"],
    "race_ethnicity":     ["black", "white", "asian", "latino", "other_race_or_ethnicity"],
    "disability":         ["physical_disability", "intellectual_or_learning_disability",
                           "psychiatric_or_mental_illness", "other_disability"],
}
IDENTITY_COLS = [c for cols in IDENTITY_CATEGORIES.values() for c in cols]
MENTION_COLS = [f"{c}_mentioned" for c in IDENTITY_COLS]
CATEGORY_COLS = [f"{cat}_mentioned" for cat in IDENTITY_CATEGORIES]

def load_data(with_splits=True):
    """Load the identity subset; if with_splits, attach each comment's train/val/test split."""
    data = pd.read_parquet(SUBSET_PATH)
    if with_splits:
        splits = pd.read_csv(SPLITS_PATH)
        data = data.merge(splits, on="id", how="inner")   # keeps only comments that were split
    return data

def get_scores(model, X):
    """Probability of toxic for LR/NB; signed distance from the boundary for LinearSVC."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def best_f1_threshold(y_true, scores):
    """Threshold that maximizes F1. Only ever call this on validation data."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    best = np.argmax(f1[:-1])   # the last precision/recall pair has no threshold
    return thresholds[best], f1[best]

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
print("Files in project folder:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
Files in project folder: ['all_data.csv', 'identity_individual_annotations.csv', 'sample_submission.csv', 'test.csv', 'test_private_expanded.csv', 'test_public_expanded.csv', 'toxicity_individual_annotations.csv', 'train.csv', 'identity_subset.parquet', 'splits.csv', 'tfidf_vectorizer.joblib']


In [ ]:
#This step loads the identity subset (from train.csv)
df = load_data(with_splits=False)
print(f"Loaded {len(df):,} identity-annotated comments")
print(f"Toxic rate: {df['toxic'].mean():.2%}")

Loaded 405,130 identity-annotated comments
Toxic rate: 11.36%


In [ ]:
#removes empty and duplicated comments
n_start = len(df)
df = df[df["comment_text"].notna() & (df["comment_text"].str.strip() != "")].reset_index(drop=True)
print(f"Removed {n_start - len(df):,} empty comments")

def normalize_text(s):
    s = s.lower()
    s = re.sub(r"https?://\\S+|www\\.\\S+", " ", s)   # remove URLs
    s = re.sub(r"[^a-z0-9\\s]", " ", s)              # remove punctuation and symbols
    return re.sub(r"\\s+", " ", s).strip()           # collapse whitespace

norm = df["comment_text"].map(normalize_text)
df["dedup_key"] = norm.where(norm != "", df["comment_text"])   # emoji-only comments keep their own text

dup_rows = df[df.duplicated("dedup_key", keep=False)]
labels_per_group = dup_rows.groupby("dedup_key")["toxic"].nunique()
print(f"Duplicate groups: {len(labels_per_group):,} "
      f"({(labels_per_group > 1).sum():,} where copies disagree on the toxic label)")

n_before = len(df)
df = (df.sort_values(["toxicity_annotator_count", "id"], ascending=[False, True])
        .drop_duplicates("dedup_key", keep="first")
        .sort_values("id")
        .drop(columns="dedup_key")
        .reset_index(drop=True))
print(f"Removed {n_before - len(df):,} duplicates; {len(df):,} comments remain")

Removed 2 empty comments
Duplicate groups: 1,584 (188 where copies disagree on the toxic label)
Removed 3,122 duplicates; 402,006 comments remain


In [ ]:
#Executes a stratified 70/20/10 split on the data.
#The stratified split preserves the percentage of toxic comments and the percentages for each identity within each subgroup
VAL_FRAC, TEST_FRAC = 0.20, 0.10
MIN_STRATUM = 30             # keys with fewer comments than this are folded into "other"
OVERWRITE_SPLITS = False

if os.path.exists(SPLITS_PATH) and not OVERWRITE_SPLITS:
    splits = pd.read_csv(SPLITS_PATH)
    print(f"Loaded existing split from {SPLITS_PATH}")
else:
    # Stratification key: toxic label x which identity is mentioned
    mentions = df[MENTION_COLS]
    n_mentions = mentions.sum(axis=1)
    single = mentions.idxmax(axis=1).str.replace("_mentioned", "", regex=False)
    identity_key = pd.Series(np.where(n_mentions == 0, "none",
                             np.where(n_mentions == 1, single, "other")), index=df.index)
    strat = df["toxic"].astype(str) + "_" + identity_key
    too_small = strat.map(strat.value_counts()) < MIN_STRATUM
    strat[too_small] = df.loc[too_small, "toxic"].astype(str) + "_other"

    # Split 70 / 30, then split the 30 into 20 (validation) and 10 (test)
    train_ids, temp_ids, _, strat_temp = train_test_split(
        df["id"], strat, test_size=VAL_FRAC + TEST_FRAC, stratify=strat, random_state=SEED)
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=strat_temp,
        random_state=SEED)

    splits = pd.concat([pd.DataFrame({"id": train_ids, "split": "train"}),
                        pd.DataFrame({"id": val_ids, "split": "val"}),
                        pd.DataFrame({"id": test_ids, "split": "test"})], ignore_index=True)
    splits.to_csv(SPLITS_PATH, index=False)
    print(f"Created new split and saved it to {SPLITS_PATH}")

df = df.drop(columns="split", errors="ignore").merge(splits, on="id", how="inner")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())

Created new split and saved it to /content/drive/MyDrive/COSC74_project/splits.csv
split
train    281404
val       80401
test      40201


In [ ]:
#Checks to make sure that the stratified split worked - every group has approximately consistent percentages
summary = df.groupby("split").agg(n=("id", "size"), toxic_rate=("toxic", "mean"))
summary["share_of_data"] = summary["n"] / summary["n"].sum()
display(summary.reindex(["train", "val", "test"]).round(4))

category_share = df.groupby("split")[CATEGORY_COLS + ["any_identity"]].mean().T
display(category_share[["train", "val", "test"]].round(4))

test_nontoxic = df[(df["split"] == "test") & (df["toxic"] == 0)]
n_nontoxic_test = test_nontoxic[MENTION_COLS].sum().sort_values()
n_nontoxic_test.index = n_nontoxic_test.index.str.replace("_mentioned", "", regex=False)
display(n_nontoxic_test.to_frame("non-toxic comments in test"))
print("Groups with fewer than 100 non-toxic test comments:",
      n_nontoxic_test[n_nontoxic_test < 100].index.tolist())

,n,toxic_rate,share_of_data
split,,,
train,281404,0.1136,0.7
val,80401,0.1136,0.2
test,40201,0.1136,0.1


split,train,val,test
gender_mentioned,0.1999,0.2005,0.1981
sexual_orientation_mentioned,0.0287,0.0285,0.0287
religion_mentioned,0.1571,0.1570,0.1572
race_ethnicity_mentioned,0.0960,0.0961,0.0974
disability_mentioned,0.0124,0.0123,0.0125
any_identity,0.4210,0.4210,0.4210


,non-toxic comments in test
other_disability,0
other_gender,2
other_sexual_orientation,3
intellectual_or_learning_disability,7
physical_disability,11
bisexual,23
other_religion,30
other_race_or_ethnicity,34
hindu,54
buddhist,60


Groups with fewer than 100 non-toxic test comments: ['other_disability', 'other_gender', 'other_sexual_orientation', 'intellectual_or_learning_disability', 'physical_disability', 'bisexual', 'other_religion', 'other_race_or_ethnicity', 'hindu', 'buddhist']


In [ ]:
#Separates out the training and validation data to be vectorized in the next step in preparation for TF-IDF
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)

X_train, y_train = train_df["comment_text"], train_df["toxic"]
X_val, y_val = val_df["comment_text"], val_df["toxic"]
print(f"Train: {len(X_train):,}   Validation: {len(X_val):,}")

Train: 281,404   Validation: 80,401


In [ ]:
#Fits the vectorizer on the training set only. The vectorizer converts text into arrays of numbers
TFIDF_PARAMS = dict(lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
                    min_df=5, max_df=0.9, max_features=200_000, sublinear_tf=True)

vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
X_train_vec = vectorizer.fit_transform(X_train)   # learns vocabulary + IDF from training text only
X_val_vec = vectorizer.transform(X_val)           # uses the training vocabulary; learns nothing

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"X_train_vec: {X_train_vec.shape[0]:,} comments x {X_train_vec.shape[1]:,} features")
print(f"X_val_vec:   {X_val_vec.shape[0]:,} comments x {X_val_vec.shape[1]:,} features")

Vocabulary size: 200,000
X_train_vec: 281,404 comments x 200,000 features
X_val_vec:   80,401 comments x 200,000 features


In [ ]:
# saves the tf-idf vectorizer
if hasattr(vectorizer, "stop_words_"):
    del vectorizer.stop_words_

joblib.dump(vectorizer, VECTORIZER_PATH)
print(f"Saved to {VECTORIZER_PATH} ({os.path.getsize(VECTORIZER_PATH) / 1e6:,.0f} MB)")

Saved to /content/drive/MyDrive/COSC74_project/tfidf_vectorizer.joblib (8 MB)
